In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Silver table
silver_df = spark.table(
    "workspace.retail_project.silver_sales"
)

# ------------------------------------
# 1. DIM CUSTOMER
# ------------------------------------

# Select customer attributes
customer_source = silver_df.select(
    "CUSTOMERNAME",
    "CITY",
    "STATE",
    "POSTALCODE",
    "COUNTRY",
    "TERRITORY"
).dropDuplicates()

# Validate one attribute combination per customer
customer_conflicts = (
    customer_source.groupBy("CUSTOMERNAME")
    .count()
    .filter(F.col("count") > 1)
)

if customer_conflicts.count() > 0:
    raise ValueError(
        "Some customers have conflicting attributes."
    )

# Generate surrogate customer key
customer_window = Window.orderBy("CUSTOMERNAME")

dim_customer = customer_source.withColumn(
    "CustomerKey",
    F.row_number().over(customer_window)
)

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_project.dim_customer")


# ------------------------------------
# 2. DIM PRODUCT
# ------------------------------------

product_source = silver_df.select(
    "PRODUCTCODE",
    "PRODUCTLINE",
    "MSRP"
).dropDuplicates()

product_conflicts = (
    product_source.groupBy("PRODUCTCODE")
    .count()
    .filter(F.col("count") > 1)
)

if product_conflicts.count() > 0:
    raise ValueError(
        "Some products have conflicting attributes."
    )

product_window = Window.orderBy("PRODUCTCODE")

dim_product = product_source.withColumn(
    "ProductKey",
    F.row_number().over(product_window)
)

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_project.dim_product")


# ------------------------------------
# 3. DIM DATE
# ------------------------------------



# ------------------------------------
# 3. DIM DATE
# ------------------------------------

from pyspark.sql import functions as F

# 1. Find the minimum and maximum order dates
date_range = silver_df.agg(
    F.min("ORDERDATE").alias("MinDate"),
    F.max("ORDERDATE").alias("MaxDate")
).first()

# 2. Validate the date range
if date_range is None:
    raise ValueError("No date range found")

min_date = date_range["MinDate"]
max_date = date_range["MaxDate"]

if min_date is None or max_date is None:
    raise ValueError("No valid order dates found")

print("Minimum date:", min_date)
print("Maximum date:", max_date)

# 3. Generate a continuous calendar
dim_date = spark.sql(f"""
    SELECT explode(
        sequence(
            DATE '{min_date}',
            DATE '{max_date}',
            INTERVAL 1 DAY
        )
    ) AS Date
""")

# 4. Create date dimension attributes
dim_date = (
    dim_date
    .withColumn(
        "DateKey",
        F.date_format(F.col("Date"), "yyyyMMdd").cast("int")
    )
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Day", F.dayofmonth("Date"))
)

# 5. Preview the date dimension
display(dim_date.limit(10))

# 6. Save as a Gold Delta table
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_project.dim_date")

# 7. Verify the saved table
print("DimDate rows:", dim_date.count())


# ------------------------------------
# 4. DIM ORDER STATUS
# ------------------------------------

status_window = Window.orderBy("STATUS")

dim_order_status = (
    silver_df.select("STATUS")
    .distinct()
    .withColumn(
        "StatusKey",
        F.row_number().over(status_window)
    )
)

dim_order_status.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_project.dim_order_status")


# ------------------------------------
# 5. FACT SALES
# ------------------------------------

# Add dimension keys to sales records
fact_sales = (
    silver_df
    .join(
        dim_customer.select("CUSTOMERNAME", "CustomerKey"),
        on="CUSTOMERNAME",
        how="left"
    )
    .join(
        dim_product.select("PRODUCTCODE", "ProductKey"),
        on="PRODUCTCODE",
        how="left"
    )
    .join(
        dim_order_status.select("STATUS", "StatusKey"),
        on="STATUS",
        how="left"
    )
    .withColumn(
        "DateKey",
        F.date_format("ORDERDATE", "yyyyMMdd").cast("int")
    )
    .select(
        "ORDERNUMBER",
        "ORDERLINENUMBER",
        "CustomerKey",
        "ProductKey",
        "StatusKey",
        "DateKey",
        "QUANTITYORDERED",
        "PRICEEACH",
        F.col("SALES").alias("SalesAmount")
    )
)

# Check for unmatched dimension keys
invalid_keys = fact_sales.filter(
    F.col("CustomerKey").isNull() |
    F.col("ProductKey").isNull() |
    F.col("StatusKey").isNull() |
    F.col("DateKey").isNull()
)

if invalid_keys.count() > 0:
    raise ValueError(
        "FactSales contains unmatched dimension keys."
    )

# Save fact table
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_project.fact_sales")


# ------------------------------------
# 6. VALIDATE GOLD TABLES
# ------------------------------------

print("Silver rows:", silver_df.count())
print("FactSales rows:", fact_sales.count())
print("Customers:", dim_customer.count())
print("Products:", dim_product.count())
print("Dates:", dim_date.count())
print("Order statuses:", dim_order_status.count())

assert fact_sales.count() == silver_df.count(), \
    "FactSales row count does not match Silver."

display(fact_sales.limit(10))
